In [ ]:
import os
import copy
import detectron2

import random ,cv2 ,torch
import detectron2.data.transforms as T
import numpy as np
import os, json, cv2, random

from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.data.transforms import RotationTransform

from detectron2.data import build_detection_train_loader, DatasetMapper
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

In [ ]:
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.data import build_detection_test_loader
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

# 配置模型
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = os.path.join("/home/ubuntu/output", "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # 設置測試閾值
cfg.DATASETS.TEST = ("pneumothorax_val", )  # 測試數據集的名稱

# 創建 predictor
predictor = DefaultPredictor(cfg)

# 構建測試數據集加載器
test_loader = build_detection_test_loader(cfg, "pneumothorax_val")

# 創建評估器
evaluator = COCOEvaluator("pneumothorax_val", cfg, False, output_dir="./output/")
val_results = inference_on_dataset(predictor.model, test_loader, evaluator)

print(val_results)


In [ ]:
from detectron2.structures import BoxMode
from detectron2.data import DatasetCatalog, MetadataCatalog

def get_dicts(img_dir):
    json_file = os.path.join(img_dir, "via_project.json")
    with open(json_file) as f:
        imgs_anns = json.load(f)

    dataset_dicts = []
    for image_id, v in imgs_anns["_via_img_metadata"].items():
        record = {}
        
        filename = os.path.join(img_dir, v["filename"])
        height, width = cv2.imread(filename).shape[:2]
        
        record["file_name"] = filename
        record["image_id"] = image_id
        record["height"] = height
        record["width"] = width
      
        annos = v["regions"]  
        objs = []
        for anno in annos:
        
            px = anno["shape_attributes"]["all_points_x"]
            py = anno["shape_attributes"]["all_points_y"]
            poly = [(x, y) for x, y in zip(px, py)]
            poly = [p for x in poly for p in x]  

            obj = {
                "bbox": [min(px), min(py), max(px), max(py)],
                "bbox_mode": BoxMode.XYXY_ABS,
                "segmentation": [poly],
                "category_id": 0,  
            }
            objs.append(obj)
        record["annotations"] = objs
        dataset_dicts.append(record)
    return dataset_dicts

for d in ["train", "val"]:
    DatasetCatalog.register("pneumothorax_" + d, lambda d=d: get_dicts("/home/ubuntu/right/p_train/" + d))
    MetadataCatalog.get("pneumothorax_" + d).set(thing_classes=["pneumothorax"])

my_metadata = MetadataCatalog.get("pneumothorax_train")

In [ ]:
from detectron2.engine import DefaultTrainer

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("pneumothorax_train",)
cfg.DATASETS.TEST = ("pneumothorax_val",)
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml")  # Let training initialize from model zoo
cfg.SOLVER.IMS_PER_BATCH = 2  # This is the real "batch size" commonly known to deep learning people
cfg.SOLVER.BASE_LR = 0.001  # pick a good LR
cfg.SOLVER.MAX_ITER = 8000   # 300 iterations seems good enough for this toy dataset; you will need to train longer for a practical dataset
cfg.SOLVER.STEPS = []        # do not decay learning rate
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128   # The "RoIHead batch size". 128 is faster, and good enough for this toy dataset (default: 512)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # only has one class (ballon). (see https://detectron2.readthedocs.io/tutorials/datasets.html#update-the-config-for-new-datasets)
# NOTE: this config means the number of classes, but a few popular unofficial tutorials incorrect uses num_classes+1 here.

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
trainer = DefaultTrainer(cfg) 
trainer.resume_or_load(resume=False)
trainer.train()

In [ ]:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")  # path to the model we just trained
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7   # set a custom testing threshold
predictor = DefaultPredictor(cfg)

In [ ]:
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import ColorMode
dataset_dicts = get_dicts("/home/ubuntu/right/p_train/val")
for d in random.sample(dataset_dicts, 2):
    im = cv2.imread(d["file_name"])
    outputs = predictor(im) 
    v = Visualizer(im[:, :, ::-1],
                   metadata=my_metadata, 
                   scale=0.2, 
                   instance_mode=ColorMode.IMAGE_BW
    )
    out = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    
    plt.imshow(out.get_image()[:, :, ::-1])
    plt.axis('off')  
    plt.show()

In [ ]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
evaluator = COCOEvaluator("pneumothorax_val", output_dir="./output")
val_loader = build_detection_test_loader(cfg, "pneumothorax_val")
print(inference_on_dataset(predictor.model, val_loader, evaluator))

In [ ]:
predictor = DefaultPredictor(cfg)

# 讀取圖片
image_path = "/home/ubuntu/right/p_train/test/cxrz_000583.png"
im = cv2.imread(image_path)

# 執行預測
outputs = predictor(im)

# 預測結果
print(outputs)  

metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])

# 可視化結果
v = Visualizer(im[:, :, ::-1], metadata=metadata, scale=1.2)
out = v.draw_instance_predictions(outputs["instances"].to("cpu"))

# 顯示圖片
plt.figure(figsize=(10, 10))
plt.imshow(out.get_image()[:, :, ::-1])
plt.axis('off')  
plt.show()